In [1]:
!pip install triton -q
import torch
import triton
import triton.language as tl
import time

In [2]:
%%writefile matmul_bench.cu
#include <stdio.h>
#include <stdlib.h>
#include <cublas_v2.h>

#define TILE_SIZE 32
#define N 1024

__global__ void matmulNaive(float* A, float* B, float* C) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++)
            sum += A[row * N + k] * B[k * N + col];
        C[row * N + col] = sum;
    }
}

__global__ void matmulTiled(float* A, float* B, float* C) {
    __shared__ float tileA[TILE_SIZE][TILE_SIZE];
    __shared__ float tileB[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;
    float sum = 0.0f;

    int numTiles = (N + TILE_SIZE - 1) / TILE_SIZE;
    for (int t = 0; t < numTiles; t++) {
        if (row < N && (t * TILE_SIZE + threadIdx.x) < N)
            tileA[threadIdx.y][threadIdx.x] = A[row * N + t * TILE_SIZE + threadIdx.x];
        else
            tileA[threadIdx.y][threadIdx.x] = 0.0f;

        if (col < N && (t * TILE_SIZE + threadIdx.y) < N)
            tileB[threadIdx.y][threadIdx.x] = B[(t * TILE_SIZE + threadIdx.y) * N + col];
        else
            tileB[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();
        for (int k = 0; k < TILE_SIZE; k++)
            sum += tileA[threadIdx.y][k] * tileB[k][threadIdx.x];
        __syncthreads();
    }
    if (row < N && col < N)
        C[row * N + col] = sum;
}

void printMetrics(const char* name, float ms, int n) {
    float gflops = (2.0f * n * n * n) / (ms / 1000.0f) / 1e9f;
    float pct = gflops / 65000.0f * 100.0f;
    printf("%-15s  time: %8.3f ms  GFLOPS: %8.1f  %% of T4 peak: %.2f%%\n",
           name, ms, gflops, pct);
}

int main() {
    size_t size = N * N * sizeof(float);

    float* h_A = (float*)malloc(size);
    float* h_B = (float*)malloc(size);
    float* h_C = (float*)malloc(size);

    for (int i = 0; i < N * N; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 1.0f;
    }

    float* d_A; float* d_B; float* d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float ms = 0;

    dim3 threads(TILE_SIZE, TILE_SIZE);
    dim3 blocks((N + TILE_SIZE - 1) / TILE_SIZE,
                (N + TILE_SIZE - 1) / TILE_SIZE);

    // warmup
    matmulNaive<<<blocks, threads>>>(d_A, d_B, d_C);
    cudaDeviceSynchronize();

    // naive
    cudaEventRecord(start);
    matmulNaive<<<blocks, threads>>>(d_A, d_B, d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printMetrics("naive CUDA", ms, N);

    // warmup tiled
    matmulTiled<<<blocks, threads>>>(d_A, d_B, d_C);
    cudaDeviceSynchronize();

    // tiled
    cudaEventRecord(start);
    matmulTiled<<<blocks, threads>>>(d_A, d_B, d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printMetrics("tiled CUDA", ms, N);

    // cuBLAS
    cublasHandle_t handle;
    cublasCreate(&handle);
    float alpha = 1.0f, beta = 0.0f;

    // warmup
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                N, N, N, &alpha, d_B, N, d_A, N, &beta, d_C, N);
    cudaDeviceSynchronize();

    cudaEventRecord(start);
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                N, N, N, &alpha, d_B, N, d_A, N, &beta, d_C, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printMetrics("cuBLAS", ms, N);

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);
    printf("\ncuBLAS C[0][0] = %f (should be %d)\n", h_C[0], N);

    cublasDestroy(handle);
    free(h_A); free(h_B); free(h_C);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    return 0;
}

Writing matmul_bench.cu


In [3]:
!nvcc matmul_bench.cu -o matmul_bench -lcublas && ./matmul_bench

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
naive CUDA       time:    3.928 ms  GFLOPS:    546.7  % of T4 peak: 0.84%
tiled CUDA       time:    3.174 ms  GFLOPS:    676.7  % of T4 peak: 1.04%
cuBLAS           time:    0.533 ms  GFLOPS:   4032.7  % of T4 peak: 6.20%

cuBLAS C[0][0] = 1024.000000 (should be 1024)


In [4]:
import torch
import triton
import triton.language as tl

@triton.jit
def matmul_kernel(
    A, B, C,
    N,
    BLOCK_SIZE: tl.constexpr,
):
    # which block am i
    row = tl.program_id(0)
    col = tl.program_id(1)

    # pointers to start of my row in A and col in B
    row_offset = row * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    col_offset = col * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)

    sum = tl.zeros((BLOCK_SIZE, BLOCK_SIZE), dtype=tl.float32)

    for k in range(0, N, BLOCK_SIZE):
        k_offset = k + tl.arange(0, BLOCK_SIZE)

        # load tile of A: shape (BLOCK_SIZE, BLOCK_SIZE)
        a_mask = (row_offset[:, None] < N) & (k_offset[None, :] < N)
        a = tl.load(A + row_offset[:, None] * N + k_offset[None, :],
                    mask=a_mask, other=0.0)

        # load tile of B: shape (BLOCK_SIZE, BLOCK_SIZE)
        b_mask = (k_offset[:, None] < N) & (col_offset[None, :] < N)
        b = tl.load(B + k_offset[:, None] * N + col_offset[None, :],
                    mask=b_mask, other=0.0)

        sum += tl.dot(a, b)

    # write output
    c_mask = (row_offset[:, None] < N) & (col_offset[None, :] < N)
    tl.store(C + row_offset[:, None] * N + col_offset[None, :],
             sum, mask=c_mask)


def triton_matmul(A, B):
    N = A.shape[0]
    C = torch.zeros((N, N), device='cuda', dtype=torch.float32)
    BLOCK_SIZE = 32
    grid = (N // BLOCK_SIZE, N // BLOCK_SIZE)
    matmul_kernel[grid](A, B, C, N, BLOCK_SIZE=BLOCK_SIZE)
    return C


# benchmark
N = 1024
A = torch.ones((N, N), device='cuda', dtype=torch.float32)
B = torch.ones((N, N), device='cuda', dtype=torch.float32)

# warmup
_ = triton_matmul(A, B)
torch.cuda.synchronize()

# time it
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
C = triton_matmul(A, B)
end.record()
torch.cuda.synchronize()

ms = start.elapsed_time(end)
gflops = (2 * N**3) / (ms / 1000) / 1e9
pct = gflops / 65000 * 100

print(f"Triton matmul")
print(f"time:          {ms:.3f} ms")
print(f"GFLOPS:        {gflops:.1f}")
print(f"% of T4 peak:  {pct:.2f}%")
print(f"C[0][0] = {C[0][0].item()} (should be {N})")

Triton matmul
time:          2.211 ms
GFLOPS:        971.5
% of T4 peak:  1.49%
C[0][0] = 1024.0 (should be 1024)
